# 03 — MBQC Protocol (Sketch)

**quantum-nonlocality** | Jinwon Yoo · C. Praise Anyanwu

---

> ⚠️ **This notebook is a research sketch.** The framework and helper functions are in place, but the core theoretical result — the specific measurement bases and feedforward rules that guarantee a deterministic Bell state output — is left for the research phase.

This notebook covers Phase 3:

1. Perform projective measurements on the bulk qubits
2. Read out the classical measurement outcomes
3. Apply Pauli corrections (feedforward) based on outcomes
4. Verify the boundary qubits are in the target Bell state

### The key open question
**What measurement basis on the bulk qubits deterministically produces $|\Phi^+\rangle$ or $|\Psi^-\rangle$ at the boundaries (up to known Pauli corrections)?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, state_fidelity
from qiskit.visualization import plot_histogram

simulator = AerSimulator()
print("Imports OK ✓")

## 1. Rebuild the Hybrid State

We reuse the construction from `02_hybrid_state_construction.ipynb`.

In [ ]:
def build_hybrid_state(n_bulk):
    """Build the full hybrid state: |+⟩_L ⊗ GHZ_n ⊗ |+⟩_R with CNOT stitching."""
    total = n_bulk + 2
    qc = QuantumCircuit(total)
    qc.h(0)
    qc.h(total - 1)
    qc.barrier()
    qc.h(1)
    for i in range(2, n_bulk + 1):
        qc.cx(1, i)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(total - 1, n_bulk)
    qc.barrier()
    return qc


def target_bell_state(name='phi_plus'):
    """Return statevector of a target Bell state."""
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    if name == 'phi_minus': qc.z(0)
    elif name == 'psi_plus': qc.x(1)
    elif name == 'psi_minus': qc.x(1); qc.z(0)
    return Statevector(qc)


print("Helper functions defined ✓")

## 2. Measurement Basis Framework

In MBQC, we measure each bulk qubit in a chosen basis. The most general single-qubit measurement basis is parameterized by angles $(\theta, \phi)$ on the Bloch sphere:

$$|m_\pm\rangle = \cos(\theta/2)|0\rangle \pm e^{i\phi}\sin(\theta/2)|1\rangle$$

Common choices:
- **Computational basis** $(\theta=0)$: $\{|0\rangle, |1\rangle\}$
- **X basis** $(\theta=\pi/2, \phi=0)$: $\{|+\rangle, |-\rangle\}$
- **Y basis** $(\theta=\pi/2, \phi=\pi/2)$: $\{|i\rangle, |-i\rangle\}$

In [ ]:
def measure_in_basis(qc, qubit, theta, phi, clbit):
    """
    Measure a qubit in the basis defined by (theta, phi).
    Rotates the qubit so that the target basis aligns with computational basis,
    then measures.
    """
    # Rotate basis to computational
    qc.rz(-phi, qubit)
    qc.ry(-theta, qubit)
    qc.measure(qubit, clbit)
    return qc


# Basis presets
BASIS = {
    'Z': (0, 0),
    'X': (np.pi/2, 0),
    'Y': (np.pi/2, np.pi/2),
}

print("Measurement basis framework ready ✓")

## 3. Full MBQC Circuit with Feedforward

The circuit structure is:
1. Build hybrid state
2. Measure all bulk qubits in chosen basis → classical bits
3. Apply Pauli corrections to boundary qubits conditioned on measurement outcomes
4. Read out boundary qubits

> **Research task:** Determine the correct `basis` and `correction_map` that guarantees the boundary qubits end up in |Φ⁺⟩ or |Ψ⁻⟩ deterministically.

In [ ]:
def mbqc_circuit(n_bulk, basis='X', apply_corrections=True):
    """
    Full MBQC circuit.
    
    Parameters
    ----------
    n_bulk : int
        Number of bulk qubits.
    basis : str
        Measurement basis for bulk qubits ('X', 'Y', 'Z').
    apply_corrections : bool
        Whether to apply Pauli feedforward corrections.
    
    Returns
    -------
    QuantumCircuit
    """
    total = n_bulk + 2
    qr = QuantumRegister(total, 'q')
    # Classical registers: bulk measurements + boundary readout
    cr_bulk     = ClassicalRegister(n_bulk, 'bulk')
    cr_boundary = ClassicalRegister(2, 'out')
    qc = QuantumCircuit(qr, cr_bulk, cr_boundary)

    # Step 1: build hybrid state
    qc.h(0)
    qc.h(total - 1)
    qc.barrier()
    qc.h(1)
    for i in range(2, n_bulk + 1):
        qc.cx(1, i)
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(total - 1, n_bulk)
    qc.barrier()

    # Step 2: measure bulk qubits in chosen basis
    theta, phi = BASIS[basis]
    for i in range(n_bulk):
        measure_in_basis(qc, i + 1, theta, phi, cr_bulk[i])
    qc.barrier()

    # Step 3: Pauli feedforward corrections on boundary qubits
    # -------------------------------------------------------
    # TODO (research): derive the correct correction rules.
    # The structure below is a placeholder showing how
    # classically conditioned Pauli gates work in Qiskit.
    # Replace with theoretically derived correction map.
    if apply_corrections:
        # Example placeholder: X correction on left boundary
        # conditioned on parity of all bulk outcomes
        # (this is NOT the correct rule — derive it!)
        with qc.if_test((cr_bulk, 1)):  # if bulk[0] == 1
            qc.x(0)                     # X on left boundary
        with qc.if_test((cr_bulk, 2)):  # if bulk[1] == 1 (for n≥2)
            qc.z(0)                     # Z on left boundary
    qc.barrier()

    # Step 4: measure boundary qubits
    qc.measure(0,          cr_boundary[0])  # left boundary
    qc.measure(total - 1,  cr_boundary[1])  # right boundary

    return qc


qc_mbqc = mbqc_circuit(n_bulk=2, basis='X', apply_corrections=False)
qc_mbqc.draw('mpl')

## 4. Simulate and Check Boundary Output

Here we use the statevector simulator to inspect what state the boundary qubits are left in *after* measuring the bulk in the X basis — before any corrections.

In [ ]:
def boundary_state_after_bulk_measurement(n_bulk, basis='X'):
    """
    For each possible bulk measurement outcome, compute the
    resulting boundary state using statevector projection.
    
    Returns a dict: outcome_bitstring -> (probability, boundary_statevector)
    """
    qc = build_hybrid_state(n_bulk)
    sv = Statevector(qc)
    
    total = n_bulk + 2
    theta, phi = BASIS[basis]
    
    # Build measurement basis vectors
    c0 = np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])   # |m+⟩
    c1 = np.array([np.sin(theta/2), -np.exp(1j*phi)*np.cos(theta/2)])  # |m-⟩
    basis_vecs = [c0, c1]
    
    results = {}
    # Iterate over all 2^n_bulk bulk measurement outcomes
    for outcome_int in range(2**n_bulk):
        outcome = format(outcome_int, f'0{n_bulk}b')
        
        # Project bulk qubits onto the outcome
        proj_state = sv.data.copy()
        # (Simplified projection — full implementation in research phase)
        # This is a placeholder showing the structure
        results[outcome] = outcome  # replace with actual projected state
    
    return results


# Simpler approach: run the circuit and look at boundary correlations
n_bulk = 2
qc_test = build_hybrid_state(n_bulk)

# Measure bulk in X basis by applying H before measurement
for i in range(1, n_bulk + 1):
    qc_test.h(i)
qc_test.measure_all()

compiled = transpile(qc_test, simulator)
counts = simulator.run(compiled, shots=8192).result().get_counts()

# Group by boundary outcomes
boundary_dist = {}
total = n_bulk + 2
for bitstr, count in counts.items():
    left  = bitstr[-1]   # q[0]
    right = bitstr[0]    # q[total-1]
    bulk  = bitstr[1:-1] # bulk qubits
    key = f"bulk={bulk[::-1]} → boundary={left}{right}"
    boundary_dist[key] = boundary_dist.get(key, 0) + count

print(f"Boundary outcomes conditioned on bulk measurement (X basis, n_bulk={n_bulk}):\n")
for k in sorted(boundary_dist):
    prob = boundary_dist[k] / 8192
    print(f"  {k}: P = {prob:.3f}")

## 5. Fidelity Check Framework

Once the correct measurement bases and correction rules are derived theoretically, use this function to verify the boundary state fidelity against the target Bell states.

In [ ]:
def check_fidelity(boundary_dm, target='phi_plus'):
    """
    Compute fidelity between the boundary density matrix
    and a target Bell state.
    
    F = 1.0 means perfect Bell state output.
    """
    target_sv = target_bell_state(target)
    target_dm = DensityMatrix(target_sv)
    F = state_fidelity(boundary_dm, target_dm)
    return F


# Example usage (placeholder — boundary_dm needs to come from
# the post-correction state once feedforward rules are derived)
print("Fidelity check function ready.")
print()
print("Usage after deriving correction rules:")
print("  F = check_fidelity(boundary_dm, target='phi_plus')")
print("  F = check_fidelity(boundary_dm, target='psi_minus')")
print()
print("Target: F = 1.0 for deterministic Bell state output.")

## 6. Open Research Questions

The following are the core theoretical tasks for this phase:

### 6.1 Derive the correct measurement basis
Which basis (or combination of bases) on the bulk qubits produces a deterministic Bell state at the boundaries?

**Starting point:** Try the X basis on all bulk qubits. Work out analytically what state the boundaries are left in for each of the $2^n$ bulk outcomes.

### 6.2 Derive the Pauli correction map
Given a bulk measurement outcome (a bitstring of length n), what Pauli gates ($I$, $X$, $Z$, $XZ$) need to be applied to each boundary qubit to recover the target Bell state?

**Expected structure:**
$$U_{\text{corr}} = \sigma_x^{s_L} \sigma_z^{t_L} \otimes \sigma_x^{s_R} \sigma_z^{t_R}$$
where $s_L, t_L, s_R, t_R \in \{0,1\}$ depend on the bulk outcomes.

### 6.3 Prove determinism
Show that for *any* bulk measurement outcome, the correction map always recovers the same target Bell state. This is the key result.

### 6.4 Generalize
Does the protocol work for arbitrary $n$? What happens to the fidelity when realistic noise is added (see Phase 4)?

---

**Next:** `04_analysis.ipynb` — Bell inequality tests, scaling with n, and noise modeling.